In [ ]:
# Install OmniVoice directly from the GitHub repository along with Gradio and other utilities
!pip install -q git+https://github.com/k2-fsa/OmniVoice.git
!pip install -q gradio accelerate pydub soundfile hf_transfer

# Set environment variable to make HuggingFace model downloads much faster
%env HF_HUB_ENABLE_HF_TRANSFER=1

In [ ]:
import logging
import os
import gc
from typing import Any, Dict

import numpy as np
import torch
from omnivoice import OmniVoice, OmniVoiceGenerationConfig
from omnivoice.cli.demo import build_demo

logging.basicConfig(
    level=logging.WARNING,
    format="%(asctime)s %(name)s %(levelname)s: %(message)s",
)
logging.getLogger("omnivoice").setLevel(logging.DEBUG)

# ---------------------------------------------------------------------------
# Model loading
# ---------------------------------------------------------------------------
CHECKPOINT = os.environ.get("OMNIVOICE_MODEL", "k2-fsa/OmniVoice")

print(f"Loading model from {CHECKPOINT} to cuda ...")

# MEMORY TIP: If you are okay with manually typing the transcript for your
# reference audio during Voice Cloning, change `load_asr` to False.
# This will prevent loading the Whisper model and save ~2-3GB of VRAM.
model = OmniVoice.from_pretrained(
    CHECKPOINT,
    device_map="cuda",
    dtype=torch.float16,
    load_asr=True,
)
sampling_rate = model.sampling_rate
print("Model loaded successfully!")

# ---------------------------------------------------------------------------
# Generation logic
# ---------------------------------------------------------------------------

# MEMORY FIX 1: Enforce inference mode so PyTorch doesn't store hidden states
@torch.inference_mode()
def generate_fn(
    text,
    language,
    ref_audio,
    instruct,
    num_step,
    guidance_scale,
    denoise,
    speed,
    duration,
    preprocess_prompt,
    postprocess_output,
    mode,
    ref_text=None,
):
    if not text or not text.strip():
        return None, "Please enter the text to synthesize."

    gen_config = OmniVoiceGenerationConfig(
        num_step=int(num_step or 32),
        guidance_scale=float(guidance_scale) if guidance_scale is not None else 2.0,
        denoise=bool(denoise) if denoise is not None else True,
        preprocess_prompt=bool(preprocess_prompt),
        postprocess_output=bool(postprocess_output),
    )

    lang = language if (language and language != "Auto") else None

    kw: Dict[str, Any] = dict(
        text=text.strip(), language=lang, generation_config=gen_config
    )

    if speed is not None and float(speed) != 1.0:
        kw["speed"] = float(speed)
    if duration is not None and float(duration) > 0:
        kw["duration"] = float(duration)

    if mode == "clone":
        if not ref_audio:
            return None, "Please upload a reference audio."
        kw["voice_clone_prompt"] = model.create_voice_clone_prompt(
            ref_audio=ref_audio,
            ref_text=ref_text,
        )

    if mode == "design":
        if instruct and instruct.strip():
            kw["instruct"] = instruct.strip()

    try:
        audio = model.generate(**kw)
        waveform = audio[0].squeeze(0).cpu().numpy() # explicitly move to CPU
        waveform = (waveform * 32767).astype(np.int16)
        result = (sampling_rate, waveform), "Done."
    except Exception as e:
        result = None, f"Error: {type(e).__name__}: {e}"
    finally:
        # MEMORY FIX 2: Force PyTorch to release cached VRAM after every generation
        gc.collect()
        torch.cuda.empty_cache()

    return result

# ---------------------------------------------------------------------------
# Build and launch demo
# ---------------------------------------------------------------------------
demo = build_demo(model, CHECKPOINT, generate_fn=generate_fn)

if __name__ == "__main__":
    demo.queue().launch(share=True, debug=True)